In [1]:
!pip install playwright bs4
!playwright install


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 MB 19.7 MB/s eta 0:00:00
(node:667) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
164.7 MiB [] 0% 0.0s164.7 MiB [] 0% 40.7s164.7 MiB [] 0% 16.6s164.7 MiB [] 0% 12.8s164.7 MiB [] 0% 7.5s164.7 MiB [] 1% 4.9s164.7 MiB [] 2% 4.2s164.7 MiB [] 2% 4.1s164.7 MiB [] 2% 4.6s164.7 MiB [] 2% 4.9s164.7 MiB [] 3% 5.3s164.7 MiB [] 3% 5.2s164.7 MiB [] 3% 5.3s164.7 MiB [] 4% 5.5s164.7 MiB [] 5% 4.6s164.7 MiB [] 6% 4.0s164.7 MiB [] 7% 3.7s164.7 MiB [] 8% 3.4s164.7 MiB [] 8% 3.2s164.7 MiB [] 9% 3.1s164.7 MiB [] 10% 2.9s164.7 MiB [] 11% 2.7s164.7 MiB [] 12% 2.5s164.7 MiB [] 14% 2.4s164.7 MiB [] 14% 2.3s164.7 MiB [] 15% 2.3s164.7 MiB [] 16% 2.2s164.7 MiB [] 17% 2.1s164.7 MiB [] 19% 2.0s164.7 MiB [] 21% 1.9s164.7 MiB

In [2]:
import os
import re
import json
from datetime import datetime
from pathlib import Path

from bs4 import BeautifulSoup
import asyncio
from playwright.async_api import async_playwright


# ==============================
# CONFIG
# ==============================
CATALOGS = [
    "https://vnexpress.net/thoi-su",
    "https://vnexpress.net/the-gioi",
    "https://vnexpress.net/kinh-doanh",
    "https://vnexpress.net/khoa-hoc-cong-nghe",
    "https://vnexpress.net/goc-nhin",
    "https://vnexpress.net/bat-dong-san",
    "https://vnexpress.net/suc-khoe",
    "https://vnexpress.net/the-thao",
    "https://vnexpress.net/giai-tri",
    "https://vnexpress.net/phap-luat",
    "https://vnexpress.net/giao-duc",
    "https://vnexpress.net/doi-song",
    "https://vnexpress.net/oto-xe-may",
    "https://vnexpress.net/du-lich",

    # "https://vnexpress.net/thoi-su/chinh-tri",
    # "https://vnexpress.net/thoi-su/chinh-tri/nhan-su",
    # "https://vnexpress.net/thoi-su/huong-toi-ky-nguyen-moi",
    # "https://vnexpress.net/thoi-su/dan-sinh",
    # "https://vnexpress.net/thoi-su/lao-dong-viec-lam",
    # "https://vnexpress.net/thoi-su/giao-thong",
    # "https://vnexpress.net/thoi-su/quy-hy-vong",

    # "https://vnexpress.net/the-gioi/phan-tich",
    # "https://vnexpress.net/the-gioi/tu-lieu",
    # "https://vnexpress.net/the-gioi/quan-su",
    # "https://vnexpress.net/the-gioi/cuoc-song-do-day",
    # "https://vnexpress.net/the-gioi/nguoi-viet-5-chau",
    # "https://vnexpress.net/the-gioi/bac-my",

    # "https://vnexpress.net/kinh-doanh/net-zero",
    # "https://vnexpress.net/kinh-doanh/quoc-te",
    # "https://vnexpress.net/kinh-doanh/doanh-nghiep",
    # "https://vnexpress.net/kinh-doanh/chung-khoan",
    # "https://vnexpress.net/kinh-doanh/ebank",
]

TOTAL_PAGES = 5  # giảm cho Colab
OUTPUT_DIR = "vnexpress_output"
DOWNLOADED_FILE = "downloaded.txt"

ARTICLE_REGEX = re.compile(
    r'https:\/\/vnexpress\.net\/[a-z0-9-]+-(\d+)\.html'
)


In [3]:
# ==============================
# UTILS giữ nguyên
# ==============================
def safe_filename(name: str):
    return re.sub(r"[^a-zA-Z0-9_.-]", "_", name)


def load_downloaded():
    if not os.path.exists(DOWNLOADED_FILE):
        return set()
    return {x.strip() for x in open(DOWNLOADED_FILE, "r", encoding="utf-8")}


def save_downloaded(url: str):
    with open(DOWNLOADED_FILE, "a", encoding="utf-8") as f:
        f.write(url + "\n")



In [4]:
# ==============================
# LINK SCRAPER (thay page->request)
# ==============================
async def extract_article_links(html: str):
    return sorted(set(m.group(0) for m in ARTICLE_REGEX.finditer(html)))


async def crawl_catalog_page(request, url):
    print(f"[i] Đang crawl: {url}")

    resp = await request.get(url)
    html = await resp.text()

    links = await extract_article_links(html)
    print(f"[✔] {url} → {len(links)} bài")
    return links



In [5]:
# ==============================
# ARTICLE PARSER giữ nguyên
# ==============================
def parse_article(html: str):
    soup = BeautifulSoup(html, "html.parser")

    title_el = soup.select_one("h1.title-detail")
    title = title_el.get_text(strip=True) if title_el else "No title"

    time_el = soup.select_one("span.date")
    published_time = time_el.get_text(strip=True) if time_el else ""

    # Content
    body_el = soup.select_one("article.fck_detail")
    paragraphs = []
    if body_el:
        for p in body_el.select("p"):
            text = p.get_text(" ", strip=True)
            if text:
                paragraphs.append(text)
    content = "\n\n".join(paragraphs)

    # Images
    images = []
    for img in soup.select("article.fck_detail img"):
        src = img.get("src") or img.get("data-src") or ""

        # ❗ FIX: bỏ qua ảnh dạng data:
        if not src or src.startswith("data:"):
            continue

        if src.startswith("//"):
            src = "https:" + src

        images.append(src)

    og = soup.select_one('meta[property="og:image"]')
    if og:
        og_url = og.get("content")
        if og_url and not og_url.startswith("data:"):
            images.insert(0, og_url)

    return {
        "title": title,
        "published_time": published_time,
        "content": content,
        "images": images,
    }



In [6]:
# ==============================
# IMAGE DOWNLOADER giữ nguyên logic
# ==============================
async def download_image(request, url: str, save_path: Path):
    if save_path.exists():
        return save_path.stat().st_size

    try:
        resp = await request.get(url)
        if resp.status != 200:
            print(f"[!] Failed image: {url}")
            return None

        data = await resp.body()
        save_path.parent.mkdir(parents=True, exist_ok=True)
        save_path.write_bytes(data)

        print(f"[+] Download {url} -> {save_path}")
        return len(data)

    except Exception as e:
        print(f"[!] Image error: {e}")
        return None




In [7]:
# ==============================
# SAVE MARKDOWN + JSON (NGUYÊN BẢN)
# ==============================
def make_markdown(url: str, scraped_at: str, data, downloaded_files):
    md = []
    md.append("# VnExpress Article\n")
    md.append(f"- **URL**: {url}")
    md.append(f"- **Scraped**: {scraped_at}")
    md.append(f"- **Title**: {data['title']}")
    md.append(f"- **Published**: {data['published_time']}")
    md.append("")

    if downloaded_files:
        md.append("## 📷 Images\n")
        for fname in downloaded_files:
            md.append(f"- ![{fname}](images/{fname})")
        md.append("")

    md.append("## 📝 Content\n")
    md.append(data["content"])
    md.append("")
    return "\n".join(md)


def write_json(path: Path, url: str, scraped_at: str, data, downloaded_files):
    output = {
        "url": url,
        "scraped_at": scraped_at,
        "title": data["title"],
        "published_time": data["published_time"],
        "content": data["content"],
        "images": [
            {"file": fname, "path": str(downloaded_files[fname]["path"])}
            for fname in downloaded_files
        ],
    }
    path.write_text(json.dumps(output, indent=4, ensure_ascii=False), encoding="utf-8")


In [8]:
# ==============================
# SCRAPE FULL ARTICLE (y như bản gốc)
# ==============================
async def scrape_article(url: str, request):

    print(f"\n>>> Scraping article: {url}")

    resp = await request.get(url)
    html = await resp.text()

    parsed = parse_article(html)

    slug = safe_filename(url.split("/")[-1].split(".")[0])
    article_dir = Path(OUTPUT_DIR) / slug
    images_dir = article_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)

    scraped_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    downloaded = {}

    for img_url in parsed["images"]:
        if img_url.startswith("data:"):
          print("[SKIP data-image]")
          continue

        fname = safe_filename(img_url.split("?")[0].split("/")[-1])
        local = images_dir / fname

        size = await download_image(request, img_url, local)

        if size:
            downloaded[fname] = {"path": local, "size": size}

    # Save Markdown (NGUYÊN BẢN)
    md = make_markdown(url, scraped_at, parsed, downloaded)
    (article_dir / "article.md").write_text(md, encoding="utf-8")

    # Save JSON (NGUYÊN BẢN)
    write_json(article_dir / "article.json", url, scraped_at, parsed, downloaded)

    print(f"[✔] Saved → {article_dir}")




In [9]:
# ==============================
# MAIN (async)
# ==============================
async def main_async():

    downloaded = load_downloaded()

    async with async_playwright() as pw:
        request = await pw.request.new_context()

        all_links = set()

        # STEP 1 — Crawl categories
        for catalog in CATALOGS:
            for i in range(1, TOTAL_PAGES + 1):
                url = catalog if i == 1 else f"{catalog}-p{i}"

                try:
                    links = await crawl_catalog_page(request, url)
                    all_links.update(links)
                except Exception as e:
                    print(f"[!] Error catalog {url}: {e}")

        print(f"\n[✔] Tổng số link tìm thấy: {len(all_links)}\n")

        # STEP 2 — Scrape articles
        for url in sorted(all_links):

            if url in downloaded:
                print(f"[SKIP] {url} (đã có trong downloaded.txt)")
                continue

            try:
                await scrape_article(url, request)
                save_downloaded(url)
            except Exception as e:
                print(f"[!] Error article {url}: {e}")



await main_async()


Streaming output truncated to the last 5000 lines.
[+] Download https://vcdn1-giaitri.vnecdn.net/2025/12/13/hongocha1jpg-1765611489-176561-1581-6219-1765611730.jpg?w=1200&h=0&q=100&dpr=1&fit=crop&s=6s9i745tleTLIu7Rn4L_Qg -> vnexpress_output/ho-ngoc-ha-cung-gigi-huong-giang-lam-moi-hit-co-don-tren-sofa-4993431/images/hongocha1jpg-1765611489-176561-1581-6219-1765611730.jpg
[+] Download https://iv1cdn.vnecdn.net/giaitri/images/web/2025/12/13/ho-ngoc-ha-lam-moi-hit-co-don-tren-sofa-1765611341.png?w=0&h=0&q=100&dpr=1&fit=crop&s=PwLhVyHoKNTzzly0AcYbIA -> vnexpress_output/ho-ngoc-ha-cung-gigi-huong-giang-lam-moi-hit-co-don-tren-sofa-4993431/images/ho-ngoc-ha-lam-moi-hit-co-don-tren-sofa-1765611341.png
[+] Download https://iv1cdn.vnecdn.net/giaitri/images/web/2022/10/19/mv-co-don-tren-sofa-ho-ngoc-ha-1666194828.jpg?w=0&h=0&q=100&dpr=1&fit=crop&s=DRaQosvfk8RBUyYnBdFE2Q -> vnexpress_output/ho-ngoc-ha-cung-gigi-huong-giang-lam-moi-hit-co-don-tren-sofa-4993431/images/mv-co-don-tren-sofa-ho-ngoc-ha

In [10]:
import os
import zipfile
import math
from google.colab import drive

drive.mount('/content/drive')

# ============================
# CONFIG
# ============================
source_folder = "vnexpress_output"             # thư mục cần nén
output_dir = "/content/drive/My Drive/scrapping" # nơi lưu zip
chunk_size_mb = 500                            # mỗi file zip tối đa 500 MB
chunk_size = chunk_size_mb * 1024 * 1024       # đổi sang bytes

os.makedirs(output_dir, exist_ok=True)



Mounted at /content/drive


In [11]:
# ============================
# Tính tổng dung lượng folder
# ============================
total_size = 0
for root, dirs, files in os.walk(source_folder):
    for file in files:
        fp = os.path.join(root, file)
        total_size += os.path.getsize(fp)

print(f"Tổng dung lượng thư mục: {total_size/1024/1024:.2f} MB")

# ============================
# Tính số lượng zip cần chia
# ============================
num_parts = math.ceil(total_size / chunk_size)
print(f"Số file ZIP cần tạo: {num_parts}")



Tổng dung lượng thư mục: 1478.51 MB
Số file ZIP cần tạo: 3


In [12]:
# ============================
# Tiến hành nén và chia file
# ============================
current_zip_index = 1
current_zip_size = 0

zip_path = os.path.join(output_dir, f"vnexpress_part_{current_zip_index}.zip")
zipf = zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED)

for root, dirs, files in os.walk(source_folder):
    for file in files:
        file_path = os.path.join(root, file)

        # nếu thêm file này vượt quá 500MB → tạo zip mới
        file_size = os.path.getsize(file_path)
        if current_zip_size + file_size > chunk_size:
            zipf.close()
            current_zip_index += 1

            zip_path = os.path.join(output_dir, f"vnexpress_part_{current_zip_index}.zip")
            zipf = zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED)
            current_zip_size = 0

        # Thêm file vào zip hiện tại
        arcname = os.path.relpath(file_path, start=source_folder)
        zipf.write(file_path, arcname)
        current_zip_size += file_size

zipf.close()

print("Hoàn thành chia ZIP vào MyDrive/scraping/")


Hoàn thành chia ZIP vào MyDrive/scraping/
